# ROBUST04 Run 3: Advanced Score Fusion + Query-Adaptive Weighting

## Strategy
- Combine multiple retrieval methods using normalized score fusion
- Methods:
  1. BM25 (multiple parameter settings)
  2. BM25 + RM3 (multiple parameter settings)
  3. Query Language Model (Dirichlet)
  4. BM25 + RM3 with aggressive expansion
- Use min-max normalization for score alignment
- Apply query-adaptive weighting based on query characteristics
- Expected MAP: 0.28-0.30+

In [ ]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)
# ============================================================================

In [ ]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Uninstall conflicting versions first
!pip uninstall -y pyserini transformers -q

# Install specific compatible versions
!pip install -q transformers==4.41.2
!pip install -q pyserini==0.22.1

# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers accelerate google-generativeai #numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")

In [ ]:
import os
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
import numpy as np
import math

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP - Add this as the first cell
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory (CHANGE THIS to match your folder!)
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    # Check if directory exists
    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        print(f"  Please update BASE_DIR to match your folder")
else:
    # Running locally
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files...")
if os.path.exists('queriesROBUST.txt'):
    print(f"  ✓ Found: queriesROBUST.txt")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")

if os.path.exists('qrels_50_Queries'):
    print(f"  ✓ Found: qrels_50_Queries")
else:
    print(f"  ✗ Missing: qrels_50_Queries")

print("\n" + "="*70)
print("Setup complete! Continue with the notebook below.")
print("="*70)


## 1. Load ROBUST04 Index

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# Create multiple searchers for different parameter configurations
searcher1 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25 conservative
searcher2 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25 aggressive
searcher3 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25+RM3 conservative
searcher4 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25+RM3 aggressive
searcher5 = LuceneSearcher.from_prebuilt_index('robust04')  # QL Dirichlet

print(f"Index contains {index_reader.stats()['documents']} documents")

## 2. Load Queries and QRELs

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                qid, query_text = parts
                queries[qid] = query_text
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                docid = parts[2]
                rel = int(parts[3])
                qrels[qid][docid] = rel
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')

train_qids = sorted(qrels.keys())
test_qids = sorted([qid for qid in queries.keys() if qid not in train_qids])

print(f"Loaded {len(queries)} queries")
print(f"Training queries: {len(train_qids)}")
print(f"Test queries: {len(test_qids)}")

## 3. Configure Searchers with Different Parameters

In [ ]:
# Method 1: BM25 Conservative (higher b for length normalization)
searcher1.set_bm25(k1=0.9, b=0.9)
print("Method 1: BM25 Conservative (k1=0.9, b=0.9)")

# Method 2: BM25 Aggressive (lower b, higher k1)
searcher2.set_bm25(k1=1.8, b=0.4)
print("Method 2: BM25 Aggressive (k1=1.8, b=0.4)")

# Method 3: BM25 + RM3 Conservative
searcher3.set_bm25(k1=1.2, b=0.75)
searcher3.set_rm3(fb_docs=10, fb_terms=40, original_query_weight=0.7)
print("Method 3: BM25+RM3 Conservative (fb_docs=10, fb_terms=40, weight=0.7)")

# Method 4: BM25 + RM3 Aggressive (more expansion)
searcher4.set_bm25(k1=0.9, b=0.4)
searcher4.set_rm3(fb_docs=30, fb_terms=100, original_query_weight=0.3)
print("Method 4: BM25+RM3 Aggressive (fb_docs=30, fb_terms=100, weight=0.3)")

# Method 5: Query Language Model with Dirichlet smoothing
searcher5.set_qld(mu=1000)
print("Method 5: QL Dirichlet (mu=1000)")

## 4. Score Normalization Functions

In [ ]:
def min_max_normalize(scores):
    """
    Min-max normalization: (x - min) / (max - min)
    Maps scores to [0, 1] range.
    """
    if not scores:
        return {}
    
    score_values = list(scores.values())
    min_score = min(score_values)
    max_score = max(score_values)
    
    if max_score == min_score:
        return {docid: 1.0 for docid in scores}
    
    normalized = {}
    for docid, score in scores.items():
        normalized[docid] = (score - min_score) / (max_score - min_score)
    
    return normalized

def z_score_normalize(scores):
    """
    Z-score normalization: (x - mean) / std
    """
    if not scores:
        return {}
    
    score_values = list(scores.values())
    mean_score = np.mean(score_values)
    std_score = np.std(score_values)
    
    if std_score == 0:
        return {docid: 0.0 for docid in scores}
    
    normalized = {}
    for docid, score in scores.items():
        normalized[docid] = (score - mean_score) / std_score
    
    return normalized

## 5. Query Classification for Adaptive Weighting

In [ ]:
def classify_query(query_text):
    """
    Simple query classification based on length and characteristics.
    Returns: 'short', 'medium', or 'long'
    
    - Short queries (1-3 words): Likely navigational, favor exact matching (BM25)
    - Medium queries (4-6 words): Balanced, use mixed weights
    - Long queries (7+ words): Exploratory, favor expansion (RM3)
    """
    words = query_text.split()
    word_count = len(words)
    
    if word_count <= 3:
        return 'short'
    elif word_count <= 6:
        return 'medium'
    else:
        return 'long'

def get_adaptive_weights(query_text):
    """
    Get query-adaptive weights for the 5 methods.
    Returns: List of 5 weights [w1, w2, w3, w4, w5]
    """
    query_type = classify_query(query_text)
    
    if query_type == 'short':
        # Short queries: favor exact matching (BM25 methods)
        weights = [1.5, 1.2, 0.8, 0.5, 1.0]  # Favor methods 1, 2
    elif query_type == 'medium':
        # Medium queries: balanced weights
        weights = [1.0, 1.0, 1.2, 1.0, 0.8]  # Slight favor to RM3 conservative
    else:  # long
        # Long queries: favor expansion (RM3 methods)
        weights = [0.7, 0.8, 1.2, 1.5, 0.9]  # Favor methods 3, 4
    
    return weights

## 6. Multi-Method Retrieval with Score Fusion

In [ ]:
def retrieve_and_fuse(query_text, k=1000, use_adaptive=True, normalization='minmax'):
    """
    Retrieve from all methods and fuse scores.
    
    Args:
        query_text: Query string
        k: Number of documents to retrieve per method
        use_adaptive: Use query-adaptive weights
        normalization: 'minmax' or 'zscore'
    
    Returns:
        List of (docid, fused_score) sorted by score (descending)
    """
    # Retrieve from all methods
    results = [
        searcher1.search(query_text, k=k),
        searcher2.search(query_text, k=k),
        searcher3.search(query_text, k=k),
        searcher4.search(query_text, k=k),
        searcher5.search(query_text, k=k),
    ]
    
    # Convert to score dictionaries
    score_dicts = []
    for hits in results:
        scores = {hit.docid: hit.score for hit in hits}
        score_dicts.append(scores)
    
    # Normalize scores
    normalized_scores = []
    for scores in score_dicts:
        if normalization == 'zscore':
            norm_scores = z_score_normalize(scores)
        else:  # minmax
            norm_scores = min_max_normalize(scores)
        normalized_scores.append(norm_scores)
    
    # Get weights
    if use_adaptive:
        weights = get_adaptive_weights(query_text)
    else:
        weights = [1.0, 1.0, 1.0, 1.0, 1.0]
    
    # Combine scores
    fused_scores = defaultdict(float)
    all_docids = set()
    
    for norm_scores in normalized_scores:
        all_docids.update(norm_scores.keys())
    
    for docid in all_docids:
        for i, norm_scores in enumerate(normalized_scores):
            if docid in norm_scores:
                fused_scores[docid] += weights[i] * norm_scores[docid]
    
    # Sort by fused score
    sorted_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_results

## 7. Evaluation Function

In [ ]:
def evaluate_fusion(query_dict, qrels_dict, use_adaptive=True, normalization='minmax', top_k=1000):
    """
    Evaluate score fusion on given queries.
    """
    run_results = []
    
    for qid in query_dict:
        if qid not in qrels_dict:
            continue
        
        query_text = query_dict[qid]
        
        # Retrieve and fuse
        fused_results = retrieve_and_fuse(query_text, k=top_k, use_adaptive=use_adaptive, normalization=normalization)
        
        # Take top k
        for rank, (docid, score) in enumerate(fused_results[:top_k], start=1):
            run_results.append(ir_measures.ScoredDoc(
                query_id=qid,
                doc_id=docid,
                score=score
            ))
    
    # Convert qrels
    qrels_list = []
    for qid, doc_rels in qrels_dict.items():
        for docid, rel in doc_rels.items():
            qrels_list.append(ir_measures.Qrel(
                query_id=qid,
                doc_id=docid,
                relevance=rel
            ))
    
    # Evaluate
    metrics = [MAP, nDCG@10, P@10]
    results = ir_measures.calc_aggregate(metrics, qrels_list, run_results)
    
    return results

## 8. Tune Fusion Parameters on Training Set

In [ ]:
train_queries = {qid: queries[qid] for qid in train_qids}

# Test different configurations
configs = [
    {'use_adaptive': False, 'normalization': 'minmax', 'name': 'Equal weights + MinMax'},
    {'use_adaptive': True, 'normalization': 'minmax', 'name': 'Adaptive weights + MinMax'},
    {'use_adaptive': False, 'normalization': 'zscore', 'name': 'Equal weights + Z-score'},
    {'use_adaptive': True, 'normalization': 'zscore', 'name': 'Adaptive weights + Z-score'},
]

print("Tuning fusion parameters...\n")
best_config = None
best_map = 0

for config in configs:
    print(f"Testing: {config['name']}")
    results = evaluate_fusion(
        train_queries, 
        qrels, 
        use_adaptive=config['use_adaptive'],
        normalization=config['normalization']
    )
    
    map_score = results[MAP]
    print(f"  MAP: {map_score:.4f}")
    print(f"  nDCG@10: {results[nDCG@10]:.4f}")
    print(f"  P@10: {results[P@10]:.4f}\n")
    
    if map_score > best_map:
        best_map = map_score
        best_config = config

print(f"Best configuration: {best_config['name']}")
print(f"Best MAP: {best_map:.4f}")

## 9. Generate Run 3 for Test Queries (199 queries)

In [ ]:
print(f"Generating run_3.res using advanced score fusion...")
print(f"Configuration: {best_config['name']}")
print(f"Number of test queries: {len(test_qids)}")

run3_results = []

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    
    # Retrieve and fuse with best configuration
    fused_results = retrieve_and_fuse(
        query_text,
        k=1000,
        use_adaptive=best_config['use_adaptive'],
        normalization=best_config['normalization']
    )
    
    # Store top 1000 results
    for rank, (docid, score) in enumerate(fused_results[:1000], start=1):
        run3_results.append({
            'qid': qid,
            'docid': docid,
            'rank': rank,
            'score': score
        })
    
    if i % 20 == 0:
        print(f"Processed {i}/{len(test_qids)} queries")

print(f"\nTotal results: {len(run3_results)}")

## 10. Save Run 3 in TREC Format

In [ ]:
def save_trec_run(results, output_file, run_tag):
    with open(output_file, 'w') as f:
        for result in results:
            f.write(f"{result['qid']} Q0 {result['docid']} {result['rank']} {result['score']:.6f} {run_tag}\n")
    print(f"Saved {len(results)} results to {output_file}")

# Save Run 3
save_trec_run(run3_results, 'run_3.res', 'run3')

# Verify file format
print("\nFirst 5 lines of run_3.res:")
with open('run_3.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

## 11. Query Type Analysis

In [ ]:
# Analyze query distribution
query_type_counts = {'short': 0, 'medium': 0, 'long': 0}

for qid in test_qids:
    query_text = queries[qid]
    qtype = classify_query(query_text)
    query_type_counts[qtype] += 1

print("\nQuery Type Distribution (Test Set):")
for qtype, count in query_type_counts.items():
    print(f"  {qtype.capitalize()}: {count} ({100*count/len(test_qids):.1f}%)")

## 12. Summary Statistics

In [ ]:
print("\n" + "="*60)
print("RUN 3 SUMMARY - ADVANCED SCORE FUSION")
print("="*60)
print(f"Best MAP on training set (50 queries): {best_map:.4f}")
print(f"\nRetrieval Methods (5 total):")
print(f"  1. BM25 Conservative (k1=0.9, b=0.9)")
print(f"  2. BM25 Aggressive (k1=1.8, b=0.4)")
print(f"  3. BM25+RM3 Conservative (fb_docs=10, weight=0.7)")
print(f"  4. BM25+RM3 Aggressive (fb_docs=30, weight=0.3)")
print(f"  5. Query Language Model (Dirichlet, mu=1000)")
print(f"\nFusion Configuration:")
print(f"  Name: {best_config['name']}")
print(f"  Adaptive weights: {best_config['use_adaptive']}")
print(f"  Normalization: {best_config['normalization']}")
print(f"\nTest queries processed: {len(test_qids)}")
print(f"Total documents ranked: {len(run3_results)}")
print(f"Average docs per query: {len(run3_results)/len(test_qids):.1f}")
print(f"\nOutput file: run_3.res")
print("="*60)

## 13. Compare Sample Query Results (Optional)

In [ ]:
# Sample comparison for one test query
sample_qid = test_qids[0]
sample_query = queries[sample_qid]

print(f"Sample Query Analysis:")
print(f"QID: {sample_qid}")
print(f"Query: {sample_query}")
print(f"Type: {classify_query(sample_query)}")
print(f"Adaptive weights: {get_adaptive_weights(sample_query)}")

# Get top 10 from each method
print(f"\nTop 5 documents from each method:")
searchers = [searcher1, searcher2, searcher3, searcher4, searcher5]
names = ['BM25 Cons', 'BM25 Agg', 'RM3 Cons', 'RM3 Agg', 'QL']

for name, searcher in zip(names, searchers):
    hits = searcher.search(sample_query, k=5)
    print(f"\n{name}:")
    for i, hit in enumerate(hits, 1):
        print(f"  {i}. {hit.docid} (score: {hit.score:.4f})")

# Get fused top 10
fused = retrieve_and_fuse(
    sample_query,
    k=1000,
    use_adaptive=best_config['use_adaptive'],
    normalization=best_config['normalization']
)
print(f"\nFused Results:")
for i, (docid, score) in enumerate(fused[:5], 1):
    print(f"  {i}. {docid} (score: {score:.4f})")